In [ ]:
import sys, warnings
sys.path.append('..')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_raw_data, basic_clean
from src.features.engineering import add_behavioural_features, build_features_target
from src.features.splitting import stratified_split
from src.models.pipe import build_pipeline
from src.models.trainer import cross_validate
from src.models.registry import MODEL_REGISTRY
from src.utils.metrics import evaluate_classifier

df = basic_clean(load_raw_data())
df = add_behavioural_features(df)
X, y = build_features_target(df)
X_train, X_test, y_train, y_test = stratified_split(X, y)
print(f'train {X_train.shape} | test {X_test.shape}')

## Cross-validate all 5 models (compact leaderboard)
#
Same `cross_validate` helper as the production pipeline. This step is slow
(~10–15 minutes on CPU); change `n_splits=2` for a fast local trial.

In [ ]:
num_features = X_train.select_dtypes(include='number').columns.tolist()
rows = []
for name in MODEL_REGISTRY:
    pipe = build_pipeline(name, num_features=num_features)
    summary = cross_validate(pipe, X_train, y_train, n_splits=2)
    rows.append({'model': name, **summary['mean']})
result_cv = pd.DataFrame(rows).sort_values(by='roc_auc', ascending=False).set_index('model')
result_cv.round(4)

## Train a single LightGBM model and inspect test scores

In [ ]:
pipe = build_pipeline('lightgbm', num_features=num_features)
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]
preds = pipe.predict(X_test)
pd.Series(evaluate_classifier(y_test.values, preds, proba)).round(4)

## Confusion matrix + ROC / PR curves

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('LightGBM confusion matrix')
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title('ROC')
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[2])
axes[2].set_title('Precision-Recall')
plt.tight_layout()
plt.savefig('../reports/figures/03_lightgbm_curves.png', dpi=110)
plt.show()